In [1]:
from ase import symbols
from ase import Atoms
import numpy as np
from typing import Union,List
from scipy.sparse import coo_matrix

class Structure2Hist:
    def __init__(self, 
                 composition:List[Union[int,str]], 
                 r_cut:float=7.5,
                 step_hist:float=0.1,
                 flatten:bool=False,
                 sparse:bool=False):
        
        ''' 
        Structure2Hist constructs a discription of local atomic structure of chosen atom in the structure
        as histograms of all possible pair distances and radial distribution histograms.
        
        *Args:
            composition - unique atomic species in considered structure
            which is needed to construct base histograms template
            
            r_cut - radius to cut local surrounding of choosen atom, (radius of cluster)
            max values for pair histogram is 2*r_cut, for radial - r_cut
            
            step_hist - step in histograms of pair distances,
            for histograms of radial distribution step is step_hist/2
            
            flatten - if True histograms will be represent like 1D vector,
            else - as a dictionary    
        '''
        
        self.composition = {symbols.atomic_numbers[atomic_type] if not isinstance(atomic_type, int) 
                            else atomic_type for atomic_type in set(composition)}
        

        self.r_cut = r_cut
        self.step_hist = step_hist
        self.flatten = flatten
        self.sparse = sparse
        self.pair_hist_points = np.arange(0.5, 2*self.r_cut+1e-5, self.step_hist)
        self.radial_hist_points = np.linspace(0.5, self.r_cut, len(self.pair_hist_points))

        self.base_histogram = self.create_template()
        
    
    def __repr__(self):
        return f'Structure2Hist(composition={list(self.composition)}, r_cut={self.r_cut}, step_hist={self.step_hist}), flatten={self.flatten})'
        
                 
    @staticmethod
    def get_all_pair_distances(in_matrix_1:np.ndarray, in_matrix_2:np.ndarray=None)->np.ndarray:
        '''Returns sorted array of Euclidean pair distances between in_matrix_1 and in_matrix_2'''
        flag = False
        if in_matrix_2 is None:
            in_matrix_2 = in_matrix_1.copy()
            flag = True

        coords_difference = in_matrix_1[:,None]-in_matrix_2[None,:]
        pair_distances = np.sum(coords_difference**2,axis=2)**.5
        start_point=pair_distances.shape[0]
        if flag:
            return np.sort(pair_distances.reshape(-1))[start_point::2]
        else:
            return np.sort(pair_distances.reshape(-1))
        
        
    def create_template(self):
        '''Returns template of dictionary of radial and pair histograms'''
        if all([isinstance(atom_type,str) or isinstance(atom_type,int) for atom_type in self.composition]):
            atomic_numbers = sorted(list(set(Atoms(self.composition).get_atomic_numbers())))
        else:
            raise TypeError('Incorrect atomic types, must be str or int')
        
        keys = [f'{A}_{B}'for idx,A in enumerate(atomic_numbers) for B in atomic_numbers[idx:]]+[f'R_{A}' for A in atomic_numbers]
        values = (np.zeros(self.pair_hist_points.shape[0]-1, dtype=np.int64) for i in range(len(keys)))
        return dict(zip(keys,values))
    
    
    def get_histogram(self, structure:Atoms)->dict:
        '''Returns dictionary with histograms of pair and radial distances for atomic structure'''
        if not set(structure.get_atomic_numbers())<=self.composition:
            raise ValueError('Composition of structure is not subset of Structure2Hist.composition') 
        
        histogram = self.base_histogram.copy()
        
        atomic_numbers_ = structure.get_atomic_numbers()[1:]
        positions_ = structure.get_positions()[1:]
        structure_composition_ = list(set(atomic_numbers_))
        
        for idx, atomic_type_A in enumerate(structure_composition_):
            position_A = positions_[atomic_type_A==atomic_numbers_]
            distances_A_A = Structure2Hist.get_all_pair_distances(position_A)
            
            hist_key_A_A = f'{atomic_type_A}_{atomic_type_A}'
            histogram[hist_key_A_A]=np.histogram(distances_A_A, self.pair_hist_points)[0]
            
            radial_hist_key_A = f'R_{atomic_type_A}'
            histogram[radial_hist_key_A]=np.histogram(np.sqrt((position_A**2).sum(axis=1)), self.radial_hist_points)[0]
            
            for atomic_type_B in structure_composition_[idx+1:]:
                position_B = positions_[atomic_type_B==atomic_numbers_]
                distances_A_B = Structure2Hist.get_all_pair_distances(position_A, position_B)
                
                hist_key_A_B = f'{atomic_type_A}_{atomic_type_B}'
                histogram[hist_key_A_B]=np.histogram(distances_A_B, self.pair_hist_points)[0]
                
        return histogram

    
    def centrete_structure(self, structure:Atoms, index:int)->Atoms:
        '''
        Returns a structure centered on the atom at the position specified by index,
        sorted by distances from zero coordinates.
        The structure is croped by r_cut.
        The central atom has coordinates (0, 0, 0)
        '''
        
        if not isinstance(index, int):
            raise TypeError('Incorrect index format, index must be integer')
            
        centreted_structure = structure.get_positions()-structure.get_positions()[index]
        distances = (centreted_structure**2).sum(axis=1)
        arrangment = np.argsort(distances)
        sorted_coords = centreted_structure[arrangment]
        sorted_atomic_species = structure.get_atomic_numbers()[arrangment]

        structure.set_positions(sorted_coords)
        structure.set_atomic_numbers(sorted_atomic_species)
        
        r_cut_square = self.r_cut**2
        
        del structure[distances[arrangment]>r_cut_square]
        
        return structure
    
    def preproces_positions(self, structure:Atoms, positions:Union[str, int, List[int], List[float]])->List[int]:
        '''
        Return a list of index specified by positions.
        Positions of central atoms can be determined  by string (atomic type, for example Au or Ag),
        by index or list of indexes of atom in the structure and by coordinates.
           
        If atomic species definded by string there aren't in the structure the index [0] will be returned.
        
        In the case of positions determined as coordinates [x:float, y:float, z:float]
        the index of atom in this coordinates will be returned, (atol~1e-3)
        if there aren't atom at this coordinates the index [0] will be returned. 
           
        If the positions are given by a string and the structure has more than 1 atom of that type, 
        the method will return a list of indexes for all atoms of that type.
        '''
        if isinstance(positions,str):
            atomic_number = symbols.atomic_numbers[positions]
            idx = list(map(int,np.where(structure.get_atomic_numbers()==atomic_number)[0]))
            return idx if idx else [0]
        
        elif isinstance(positions,int):
            if positions<=structure.get_positions().shape[0]:
                return [positions]
            else:
                raise IndexError('Index out of range')
        
        elif isinstance(positions,list) and all((isinstance(item,int) for item in positions)):
            if max(positions)<=structure.get_positions().shape[0]:
                return positions
            else:
                raise IndexError('Index out of range')
            
        elif len(positions)==3 and all((isinstance(item, float) for item in positions)):
            coords = structure.get_positions()
            pos = np.array(positions)
            result = np.isclose(coords - pos, np.zeros(3), atol=1e-3).all(axis=1)
            return [int(result.argmax())]
        else:
            raise TypeError('Incorrect positions type')
        
    
    def create(self, structure:Atoms, positions:Union[str, int, List[float]])->dict:
        '''
        Returns list of dictionaries with histograms of pair and radial distances for atomic structure,
        which is centered on the atom specified by positions
           
        positions of central atoms can be determined  by string (atomic type, for example Au or Ag),
        by index of list of index of atom in the structure and by coordinates. 
        If the positions are given by a string and the structure has more than 1 atom of that type, 
        the method will return a list of dictionaries for all atoms of that type.
        '''
        indexes = self.preproces_positions(structure=structure.copy(), positions=positions)
        result = []
        for index in indexes:
            structure_centreted = self.centrete_structure(structure=structure.copy(), index = index)
            structure_centreted.set_positions(np.round(structure_centreted.get_positions(),5))

            
            histogram = self.get_histogram(structure_centreted)
            if self.flatten:
                histogram = np.reshape(list(histogram.values()),-1)

            if self.sparse:
                hist = coo_matrix(histogram)
                result.append(np.column_stack((hist.data,hist.col,hist.row)))
                max_val_result_shape = max(list(map(lambda x:x.shape[0],result)))
                tmplt = np.zeros((len(result),max_val_result_shape,3))
                for i,res in enumerate(result):
                    tmplt[i,:res.shape[0],:] = res
            else:
                result.append(histogram)

        return tmplt if self.sparse else result